# 07 · Multimorbidity phenotypes

Builds the comorbidity co-occurrence network, assigns each patient in the landmark population a phenotype, and tests whether phenotype is related to contact class or changes the class–outcome association.

**Reads (`DATA_DIR`):** `landmark_build` (02), `covariates` (01, Elixhauser categories), `analysis_frame` and `mi` (06); and `results/spline_knots.json`

**Produces (patient-level, `DATA_DIR`):** `phenotypes`

**Produces (aggregate, `results/`):** `multimorbidity_nodes.csv`, `multimorbidity_edges.csv`, `multimorbidity_phenotypes.csv`, `07_multimorbidity.json`

**Paper items:** Supplementary Methods and Supplementary Results, *Multimorbidity*.

**Runtime:** a few minutes. Run notebook 06 first.

| Step | What it does |
|---|---|
| 1 | Network cohort: at-index cohort, alive at day 730 with a record after day 730 |
| 2 | Nodes: Elixhauser categories (Quan coding) other than hypertension present in ≥ 1% of patients; edges: phi coefficient > 0.10 |
| 3 | Louvain community detection; communities with more than one category are the phenotypes |
| 4 | Each landmark patient is assigned to the community with the most member categories (ties: malignancy, then neuro-psychiatric-hepatic, then cardio-renal-metabolic); no member category = low-structure reference |
| 5 | χ² test and Cramér's V (phenotype × class); likelihood-ratio test for 12 phenotype-by-class terms in the cause-specific Cox model |

## 0 · Setup

In [ ]:
# ---- edit these ----
REPO_DIR = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
DATA_DIR = "/content/drive/MyDrive/htn_bp_data"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install lifelines==0.30.3

import os, sys, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *
from landmark_models import *

import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import chi2, chi2_contingency
from lifelines import CoxPHFitter

RESULTS_DIR = os.path.join(REPO_DIR, "results")

lb  = load_df(DATA_DIR, "landmark_build")
cov = load_df(DATA_DIR, "covariates")
A   = load_df(DATA_DIR, "analysis_frame").reset_index(drop=True)
MI  = load_df(DATA_DIR, "mi")
KN  = json.load(open(os.path.join(RESULTS_DIR, "spline_knots.json")))
assert (MI[MI[".imp"] == 1].subject_id.values == A.subject_id.values).all()
print(f"at-index cohort {len(lb):,} | landmark population {len(A):,}")

## 1 · Network cohort and nodes

Elixhauser categories from notebook 01 (comorbidipy, Quan coding; diagnoses on or before index). The two hypertension categories are left out because every patient has hypertension.

In [ ]:
ELIX = ["chf", "carit", "valv", "pcd", "pvd", "hypunc", "hypc", "para", "ond", "cpd", "diabunc", "diabc",
        "hypothy", "rf", "ld", "pud", "aids", "lymph", "metacanc", "solidtum", "rheumd", "coag", "obes",
        "wloss", "fed", "blane", "dane", "alcohol", "drug", "psycho", "depre"]
HTN_CATS = ["hypunc", "hypc"]
CATS = [c for c in ELIX if c in cov.columns and c not in HTN_CATS]
missing = [c for c in ELIX if c not in cov.columns]
assert not missing, f"Elixhauser columns not found in covariates: {missing}"

net_ids = lb.loc[lb.status730 == "alive: record after day 730", "subject_id"]
Xn = (cov.set_index("subject_id").loc[net_ids, CATS].fillna(0) > 0).astype(int)
prev = Xn.mean()
nodes = prev[prev >= 0.01].index.tolist()
print(f"network cohort: {len(Xn):,} patients")
print(f"categories present in >= 1%: {len(nodes)} of {len(CATS)}")
print("left out (< 1%):", sorted(set(CATS) - set(nodes)))

## 2 · Edges and communities

Phi coefficient = Pearson correlation of two 0/1 indicators. Louvain community detection on the phi-weighted graph (networkx, seed 42).

In [ ]:
PHI_CUT = 0.10
phi = pd.DataFrame(np.corrcoef(Xn[nodes].values.T), index=nodes, columns=nodes)
edges = [(a, b, float(phi.loc[a, b])) for i, a in enumerate(nodes) for b in nodes[i + 1:] if phi.loc[a, b] > PHI_CUT]
G = nx.Graph(); G.add_nodes_from(nodes); G.add_weighted_edges_from(edges)
comms = nx.community.louvain_communities(G, weight="weight", seed=42)
multi = [sorted(c) for c in comms if len(c) > 1]
print(f"edges with phi > {PHI_CUT}: {len(edges)} | communities: {len(comms)} | with more than one category: {len(multi)}")

def name_of(members):
    if {"metacanc", "solidtum"} & set(members): return "Malignancy"
    if "chf" in members:                       return "Cardio-renal-metabolic"
    return "Neuro-psychiatric-hepatic"
PH = {}
for c in multi:
    nm = name_of(c)
    assert nm not in PH, f"two communities would both be called {nm}: {PH[nm]} and {c}"
    PH[nm] = c
for nm, c in PH.items():
    print(f"  {nm}: {c}")

# communities reported in the paper, for comparison
PAPER = {"Malignancy": ["metacanc", "solidtum"],
         "Neuro-psychiatric-hepatic": ["alcohol", "coag", "depre", "drug", "fed", "ld", "ond", "para", "wloss"],
         "Cardio-renal-metabolic": ["carit", "chf", "cpd", "diabc", "obes", "pcd", "pvd", "rf", "valv"]}
for nm in PAPER:
    got = PH.get(nm, [])
    print(f"{'OK ' if sorted(got) == sorted(PAPER[nm]) else 'DIFF'} {nm}: "
          f"added {sorted(set(got) - set(PAPER[nm]))}, dropped {sorted(set(PAPER[nm]) - set(got))}")

## 3 · Phenotype for each patient in the landmark population

In [ ]:
PRIORITY = ["Malignancy", "Neuro-psychiatric-hepatic", "Cardio-renal-metabolic"]
PHENOS = ["Low structure"] + [p for p in reversed(PRIORITY) if p in PH]
cv  = (cov.set_index("subject_id").loc[A.subject_id, CATS].fillna(0) > 0)
cnt = pd.DataFrame({k: cv[PH[k]].sum(axis=1).values for k in PRIORITY if k in PH})
best = cnt.max(axis=1)
pheno = pd.Series("Low structure", index=A.index)
for k in reversed([p for p in PRIORITY if p in PH]):     # last assignment wins, so the first in PRIORITY wins ties
    pheno[(best > 0) & (cnt[k] == best)] = k
A["pheno"] = pheno.values

E = (A.status == 1).astype(int)
tab = pd.DataFrame({"n": A.pheno.value_counts(), "events": A[E == 1].pheno.value_counts()}).reindex(PHENOS).fillna(0).astype(int)
print(tab.to_string())
print("paper (October run): Low structure 3,134 (68), Cardio-renal-metabolic 5,921 (385), "
      "Neuro-psychiatric-hepatic 4,707 (158), Malignancy 1,014 (34)")

## 4 · Phenotype and contact class

χ² test and Cramér's V for phenotype by class. Effect modification: likelihood-ratio test comparing the cause-specific Cox model of notebook 06 (covariates, class, phenotype) with and without the 12 phenotype-by-class terms, on the first imputed data set.

In [ ]:
ct = pd.crosstab(A.pheno, A.label).reindex(index=PHENOS, columns=ORDER)
x2, p_chi, dof, _ = chi2_contingency(ct)
V = float(np.sqrt(x2 / (len(A) * (min(ct.shape) - 1))))
print(ct.to_string())
print(f"\nchi2 = {x2:.1f}, df = {dof}, p = {p_chi:.2g}, Cramer's V = {V:.2f}")

F1 = A.copy(); F1[MISS] = MI.loc[MI[".imp"] == 1, MISS].values
T = A.time.astype(float).values

def fit(expo):
    d = pd.concat([pd.DataFrame({"T": T, "E": E.values}), build_X(F1, KN), expo], axis=1)
    d = d.loc[:, d.std() > 0]
    return CoxPHFitter(penalizer=1e-6).fit(d, "T", "E")

Cl = class_dummies(A)
Pd = pd.get_dummies(pd.Categorical(A.pheno, categories=PHENOS), dtype=float).drop(columns="Low structure").set_index(A.index)
inter = pd.DataFrame({f"{c} x {q}": Cl[c] * Pd[q] for c in Cl for q in Pd}, index=A.index)
m0 = fit(pd.concat([Cl, Pd], axis=1))
m1 = fit(pd.concat([Cl, Pd, inter], axis=1))
lr = 2 * (m1.log_likelihood_ - m0.log_likelihood_)
df_int = inter.shape[1]
p_int = float(chi2.sf(lr, df_int))
ev = (A[E == 1].groupby(["label", "pheno"]).size().unstack(fill_value=0)
        .reindex(index=ORDER[1:], columns=Pd.columns, fill_value=0))
small = int((ev < 10).sum().sum())
print(f"interaction LR chi2 = {lr:.1f}, df = {df_int}, p = {p_int:.2f}")
print(f"interaction cells with fewer than 10 events: {small} of {ev.size}")

## 5 · Results summary and save

In [ ]:
print("MULTIMORBIDITY")
print(f"  network cohort {len(Xn):,}; {len(nodes)} categories; {len(edges)} edges; {len(multi)} communities with > 1 category")
for k in PHENOS:
    print(f"  {k:28s} n = {tab.loc[k, 'n']:,}  events = {tab.loc[k, 'events']}")
print(f"  Cramer's V = {V:.2f}; interaction p = {p_int:.2f}; {small} of {ev.size} interaction cells with < 10 events")

save_df(A[["subject_id", "pheno"]], DATA_DIR, "phenotypes")

comm_of = {c: k for k, v in PH.items() for c in v}
pd.DataFrame({"category": nodes, "prevalence_pct": (100 * prev[nodes]).round(2).values,
              "community": [comm_of.get(c, "single-category") for c in nodes]}) \
  .to_csv(os.path.join(RESULTS_DIR, "multimorbidity_nodes.csv"), index=False)
pd.DataFrame(edges, columns=["category_a", "category_b", "phi"]).round(3) \
  .to_csv(os.path.join(RESULTS_DIR, "multimorbidity_edges.csv"), index=False)
pd.concat([tab, ct], axis=1).reset_index(names="phenotype") \
  .to_csv(os.path.join(RESULTS_DIR, "multimorbidity_phenotypes.csv"), index=False)
save_json({"network_cohort_n": int(len(Xn)), "n_nodes": len(nodes), "n_edges": len(edges), "phi_cut": PHI_CUT,
           "communities": PH, "matches_paper_communities": {k: sorted(PH.get(k, [])) == sorted(v) for k, v in PAPER.items()},
           "phenotype_n": {k: int(v) for k, v in tab.n.items()}, "phenotype_events": {k: int(v) for k, v in tab.events.items()},
           "chi2": float(x2), "chi2_df": int(dof), "chi2_p": float(p_chi), "cramers_v": V,
           "interaction_lr": float(lr), "interaction_df": int(df_int), "interaction_p": p_int,
           "interaction_cells_lt10_events": small}, RESULTS_DIR, "07_multimorbidity")